# 12 · Eval, canary e rollback

Ultimo loop: confrontare baseline/candidato, bloccare regressioni, distribuire canary
stabile e conservare rollback. Solo standard library, nessuna chiamata modello.

## Obiettivi, prerequisiti e modalità di lettura

Confronterai baseline/candidato, canary e rollback. Durata: 25–35 minuti. Tutto gira offline e produce risultati deterministici.

Ogni blocco di codice è preceduto da una spiegazione e seguito da un **output
atteso**. Quando interviene un modello, l'output atteso descrive proprietà e
invarianti, non una frase letterale. Esegui le celle in ordine e non saltare i
casi negativi: mostrano il confine del meccanismo, non un incidente del corso.

## 1 · Casi verificabili e confronto paired

### Spiegazione del blocco · Risultati paired

Baseline e candidato condividono gli stessi case id, così ogni regressione è attribuibile. Le metriche aggregate non sostituiscono il confronto caso per caso.

In [ ]:
from dataclasses import dataclass
from hashlib import sha256
from statistics import mean

@dataclass(frozen=True)
class Result:
    case: str
    passed: bool
    tokens: int
    latency_ms: int

baseline = [Result("a", True, 100, 120), Result("b", False, 150, 200), Result("c", True, 120, 150)]
candidate = [Result("a", True, 90, 110), Result("b", True, 140, 180), Result("c", True, 100, 140)]

def summary(results):
    return {"pass_rate": mean(item.passed for item in results), "tokens": sum(item.tokens for item in results), "latency": mean(item.latency_ms for item in results)}

print("baseline", summary(baseline)); print("candidate", summary(candidate))

### Output atteso

Due dizionari: candidato con pass rate maggiore, meno token e latenza minore.

## 2 · Regression gate deterministico

### Spiegazione del blocco · Regression gate

Il gate blocca regressioni anche quando la media migliora. Impone inoltre tetti per token e latenza.

In [ ]:
def gate(base, cand) -> dict:
    b, c = summary(base), summary(cand)
    regressions = [left.case for left, right in zip(base, cand) if left.passed and not right.passed]
    passed = not regressions and c["pass_rate"] >= b["pass_rate"] and c["tokens"] <= b["tokens"] * 1.2 and c["latency"] <= b["latency"] * 1.3
    return {"passed": passed, "regressions": regressions, "token_ratio": c["tokens"] / b["tokens"]}

decision = gate(baseline, candidate)
assert decision["passed"]
print(decision)

### Output atteso

Dizionario con `passed=True`, lista regressioni vuota e token ratio inferiore a uno.

## 3 · Canary stabile per sessione

### Spiegazione del blocco · Canary stabile

Hash della sessione assegna sempre lo stesso arm. La distribuzione su molte sessioni approssima la frazione configurata.

In [ ]:
def arm(session_id: str, fraction: float) -> str:
    bucket = int(sha256(session_id.encode()).hexdigest()[:8], 16) / 0xFFFFFFFF
    return "canary" if bucket < fraction else "baseline"

first = arm("session-42", 0.2)
assert all(arm("session-42", 0.2) == first for _ in range(10))
counts = {name: sum(arm(f"s-{i}", 0.2) == name for i in range(1_000)) for name in ("baseline", "canary")}
print(counts)

### Output atteso

Conteggi vicini a 800 baseline e 200 canary, con variazione deterministica dovuta agli hash.

## 4 · Versione attiva e rollback append-only

### Spiegazione del blocco · Promotion e rollback

Ogni modifica salva eventi append-only. Il rollback è una nuova transizione e non cancella la storia della promozione.

In [ ]:
history = []
active = {"max_tool_calls": 12}

def promote(candidate_config: dict):
    global active
    history.append({"event": "before_promotion", "config": active.copy()})
    active = candidate_config.copy()
    history.append({"event": "promoted", "config": active.copy()})

def rollback(version: dict):
    global active
    history.append({"event": "before_rollback", "config": active.copy()})
    active = version.copy()

promote({"max_tool_calls": 16})
rollback(history[0]["config"])
assert active == {"max_tool_calls": 12}
print(history)

### Output atteso

Lista di eventi e configurazione attiva tornata a `max_tool_calls=12`.

## Regola finale

Il sistema che propone non cambia rubric o soglia con cui viene giudicato: altrimenti
può aumentare il pass rate abbassando l’esame, non migliorando l’agente.

## Prova tu

Aggiungi gate live: servono almeno cinque run per arm e non-inferiorità su qualità, token e latenza.

## Laboratorio aggiuntivo

Gli esempi seguenti riusano quanto costruito sopra. Il primo amplia il caso normale; il
secondo esercita un confine, un errore o una proprietà che spesso causa bug reali.

## Esempio aggiuntivo: media migliore con una regressione

### Spiegazione del blocco

Il confronto paired impedisce che un guadagno medio nasconda un caso precedentemente corretto diventato errato.

In [ ]:
candidato_con_regressione = [Result("a", False, 50, 80), Result("b", True, 50, 80), Result("c", True, 50, 80)]
print(gate(baseline, candidato_con_regressione))

### Output atteso

`passed=False` e `regressions=['a']`, anche se token e latenza sono migliori.

## Esempio aggiuntivo: cambiare la frazione canary

### Spiegazione del blocco

La stessa funzione permette di confrontare rollout prudente e rollout più ampio mantenendo assegnazione stabile per sessione.

In [ ]:
for frazione in (0.05, 0.20, 0.50):
    selezionate = sum(arm(f"session-{i}", frazione) == "canary" for i in range(2_000))
    print(frazione, "->", selezionate)

### Output atteso

Conteggi vicini rispettivamente a 100, 400 e 1000. Sono deterministici per questi session id.

## Riepilogo e troubleshooting

Prima di proseguire, prova a spiegare con parole tue: quale stato è cambiato, quale
componente ha preso la decisione e quale prova rende osservabile l'esito.

Se una cella fallisce:

1. rileggi l'output atteso e individua la prima invariante non rispettata;
2. verifica di aver eseguito tutte le celle precedenti nello stesso kernel;
3. per i notebook live, controlla `.env`, modello disponibile e quota API;
4. riavvia il kernel solo dopo aver conservato eventuali file che vuoi ispezionare;
5. non correggere un caso negativo: l'errore previsto è parte dell'esempio.